# 3. API Keys and the OpenAI API

Some APIs are public. Other APIs require an **API key**.

An API key is a secret string that identifies your app or account. API keys are commonly used for:
- access control
- billing or usage tracking
- rate limits
- abuse prevention

In this notebook, we will:

- load an API key from `.env`
- use the key in an HTTP request header
- send a prompt to the OpenAI API
- read the model's response


## The Most Important Rule

Do not paste a real API key into a notebook, GitHub repo, slide deck, or screenshot.

Store the real key in `.env`. Share `.env.example` so other people know which variable names they need.

```text
OPENAI_API_KEY=your-key-goes-here
```


In [ ]:
import os
from pprint import pprint

import requests
# If you do not have python-dotenv installed, run: pip install python-dotenv
from dotenv import load_dotenv

TIMEOUT = 60
MODEL = "gpt-4.1-mini"

## Load the API Key

`load_dotenv()` reads the `.env` file and adds its values to Python's environment variables.

After that, `os.environ["OPENAI_API_KEY"]` gives us the key. We will confirm the key loaded without printing it.


In [ ]:
load_dotenv()

api_key = os.environ["OPENAI_API_KEY"]

print("OPENAI_API_KEY loaded")

## Build the Request

The OpenAI API is still an HTTP API. The request has the same pieces we have practiced:

- **Endpoint**: the URL we send the request to
- **Method**: `POST`, because we are sending data
- **Headers**: extra information, including the API key
- **Body**: JSON containing the model and prompt


In [ ]:
url = "https://api.openai.com/v1/responses"

headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json",
}

payload = {
    "model": MODEL,
    "input": "Answer directly in two short sentences: What is an cat?",
    "max_output_tokens": 200,
}

print("Endpoint:", url)
print("Method: POST")
print("Headers are ready. Do not print the secret key.")
pprint(payload)

## Send the Request

`requests.post()` sends the JSON body to the API.

A status code of `200` means the request worked.


In [ ]:
response = requests.post(
    url,
    headers=headers,
    json=payload,
    timeout=TIMEOUT,
)

print(response.status_code)
response.raise_for_status()

## Read the Response

The response body is JSON, so we can convert it to Python data with `.json()`.

The model's text is inside the `output` list. The helper function below pulls out the text parts.


In [ ]:
response_data = response.json()

print("Response id:", response_data["id"])
print("Model:", response_data["model"])

In [ ]:
def extract_response_text(response_data):
    pieces = []

    for item in response_data["output"]:
        for content in item.get("content", []):
            if content.get("type") == "output_text":
                pieces.append(content["text"])

    return "\n".join(pieces)

answer = extract_response_text(response_data)
print(answer)


## Put the API Call in a Function

Once the request works, we can give the whole action a name.

This kind of function is also how many AI agents use tools: the agent chooses a function, the function calls an API, and the API gives the agent access to fresh information or useful actions.


In [ ]:
def ask_openai(prompt):
    payload = {
        "model": MODEL,
        "input": prompt,
        "max_output_tokens": 200,
    }

    response = requests.post(
        url,
        headers=headers,
        json=payload,
        timeout=TIMEOUT,
    )
    response.raise_for_status()

    return extract_response_text(response.json())

print(ask_openai("Write one friendly sentence that encourages students learning APIs."))

## Quick Review

- API keys are secrets.
- `.env` keeps secrets out of the notebook.
- `load_dotenv()` loads values from `.env`.
- `Authorization` is the header that carries the API key.
- The OpenAI API request uses `POST` because we send a JSON body.
- The response is JSON that contains the model output.


## Where This Goes Next

The next step is to put these pieces together into an agent-style workflow.

One cycle can look like this:

- Python sends a prompt to an LLM through an API.
- The LLM decides whether it needs a tool.
- A tool is a function Python can run. That function might call another API, search data, read a file, or calculate something.
- Python gives the tool result back to the LLM.
- The LLM uses that new information to answer or choose the next step.

When this happens repeatedly, it becomes a loop: model call, tool call, tool result, next model call. The important idea is that the LLM does not directly do everything by itself. Your code connects the model to tools, and APIs are one common way those tools reach the outside world.
